In [1]:
import os
import random
import math
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image

from sklearn.metrics import (confusion_matrix, classification_report, roc_auc_score,
                             roc_curve, auc)
from sklearn.manifold import TSNE
from sklearn.preprocessing import label_binarize

import networkx as nx

# Explanation tools (may be installed on Kaggle if not, notebook can pip install)
try:
    from captum.attr import LayerGradCam, GuidedBackprop
    CAPTUM_OK = True
except Exception:
    CAPTUM_OK = False

try:
    from lime import lime_image
    LIME_OK = True
except Exception:
    LIME_OK = False

# Check PyG availability; install if needed
PYG_AVAILABLE = False
try:
    import torch_geometric
    from torch_geometric.data import Data as GeoData
    from torch_geometric.nn import GCNConv, global_mean_pool
    PYG_AVAILABLE = True
except Exception as e:
    print("PyG not available; installing...")
    os.system("pip install torch-geometric")
    # Re-import after install (may need kernel restart)
    import importlib
    import torch_geometric
    PYG_AVAILABLE = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if not PYG_AVAILABLE:
    print("PyG fallback to sparse manual GNN")

PyG not available; installing...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.6 MB/s eta 0:00:00
Device: cuda


In [2]:
class AlzheimerImageDataset(Dataset):
    def __init__(self, root_dir, classes=None, transform=None):
        self.root = Path(root_dir)
        self.transform = transform
        if classes is None:
            self.classes = sorted([p.name for p in self.root.iterdir() if p.is_dir()])
        else:
            self.classes = classes
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.samples = []
        for c in self.classes:
            p = self.root / c
            if not p.exists():
                continue
            for img in p.iterdir():
                if img.suffix.lower() in ['.jpg','jpeg','.png']:
                    self.samples.append((str(img), self.class_to_idx[c]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path,label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, path

In [3]:
DATA_DIR = '/kaggle/input/alzheimers-multiclass-dataset-equal-and-augmented/combined_images'
CLASS_NAMES = ['MildDemented','ModerateDemented','NonDemented','VeryMildDemented']
NUM_CLASSES = len(CLASS_NAMES)

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 8

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

dataset = AlzheimerImageDataset(DATA_DIR, classes=CLASS_NAMES, transform=transform)
print('Total samples found:', len(dataset))

# deterministic split
random.seed(42)
num = len(dataset)
n_train = int(0.7 * num)
n_val = int(0.15 * num)
n_test = num - n_train - n_val
train_ds, val_ds, test_ds = random_split(dataset, [n_train,n_val,n_test])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=2)

print(f'Train={len(train_ds)} Val={len(val_ds)} Test={len(test_ds)}')

Total samples found: 44000
Train=30799 Val=6600 Test=6601


In [4]:
class CNNBaseline(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)
        in_feat = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_feat, num_classes)
    def forward(self,x):
        return self.backbone(x)

# For DST-HGNN we will build patch-graphs per image. Here is a lightweight hybrid model.
class PatchEncoder(nn.Module):
    def __init__(self, out_dim=128, patch_size=32):
        super().__init__()
        self.patch_size = patch_size
        self.conv = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc = nn.Linear(64, out_dim)
    def forward(self, patches):
        # patches: [B, N, 3, ps, ps]
        B,N = patches.shape[0], patches.shape[1]
        x = patches.view(B*N, 3, patches.shape[-2], patches.shape[-1])
        f = self.conv(x).view(B*N, -1)
        f = self.fc(f)
        return f.view(B, N, -1)

# Optimized GNN: PyG if available, else sparse fallback
EDGE_H = 7  # 224/32
EDGE_W = 7
if PYG_AVAILABLE:
    from torch_geometric.nn import GCNConv, global_mean_pool
    class PyGGNNLayer(nn.Module):
        def __init__(self, in_dim, out_dim):
            super().__init__()
            self.conv = GCNConv(in_dim, out_dim)
        def forward(self, x, edge_index):
            B, N, D = x.shape
            x_flat = x.view(-1, D)
            # Repeat edge_index for batch (simple: assume uniform)
            batch_edges = edge_index.repeat(1, B)
            batch_node = torch.arange(B * N, device=x.device)
            h = self.conv(x_flat, batch_edges)
            return h.view(B, N, -1).relu()
    
    class DST_HGNN_PYG(nn.Module):
        def __init__(self, patch_dim=128, gnn_hidden=128, num_classes=NUM_CLASSES, patch_size=32):
            super().__init__()
            self.patch_encoder = PatchEncoder(out_dim=patch_dim, patch_size=patch_size)
            self.gnn1 = PyGGNNLayer(patch_dim, gnn_hidden)
            self.gnn2 = PyGGNNLayer(gnn_hidden, gnn_hidden)
            self.classifier = nn.Sequential(nn.Linear(gnn_hidden,256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,num_classes))
        def forward(self, patches, edge_index):
            feats = self.patch_encoder(patches)  # [B,N,D]
            h = self.gnn1(feats, edge_index)
            h = self.gnn2(h, edge_index)
            h = h.mean(dim=1)  # global mean pool
            logits = self.classifier(h)
            return logits, h
else:
    class SparseGNNLayer(nn.Module):
        def __init__(self, in_dim, out_dim):
            super().__init__()
            self.lin = nn.Linear(in_dim * 2, out_dim)  # Self + avg neighbors
        def forward(self, x, edge_index):
            B, N, D = x.shape
            src, tgt = edge_index
            neigh = torch.zeros_like(x)
            neigh.index_add_(1, tgt.unsqueeze(0).unsqueeze(-1).expand(B, -1, D), x.gather(1, src.unsqueeze(-1).unsqueeze(-1).expand(B, -1, D, 1).squeeze(-1)))
            neigh = neigh / 4.0  # Approx degree=4
            combined = torch.cat([x, neigh], dim=-1)
            out = self.lin(combined)
            return out.relu()
    
    class DST_HGNN_Sparse(nn.Module):
        def __init__(self, patch_dim=128, gnn_hidden=128, num_classes=NUM_CLASSES, patch_size=32):
            super().__init__()
            self.patch_encoder = PatchEncoder(out_dim=patch_dim, patch_size=patch_size)
            self.gnn1 = SparseGNNLayer(patch_dim, gnn_hidden)
            self.gnn2 = SparseGNNLayer(gnn_hidden, gnn_hidden)
            self.classifier = nn.Sequential(nn.Linear(gnn_hidden,256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,num_classes))
        def forward(self, patches, edge_index):
            feats = self.patch_encoder(patches)
            h = self.gnn1(feats, edge_index)
            h = self.gnn2(h, edge_index)
            h = h.mean(dim=1)
            logits = self.classifier(h)
            return logits, h

In [5]:
from torchvision.transforms import functional as TF

def image_to_patches_torch(img_tensor, patch_size=32, target_size=224):
    # Vectorized: pad/resize/full image tensor to patches
    orig_h, orig_w = img_tensor.shape[1:]
    pad_h = max(0, target_size - orig_h)
    pad_w = max(0, target_size - orig_w)
    if pad_h > 0 or pad_w > 0:
        img_tensor = F.pad(img_tensor, (0, pad_w, 0, pad_h), mode='constant', value=0)
    img_tensor = F.interpolate(img_tensor.unsqueeze(0), size=(target_size, target_size), mode='bilinear', align_corners=False).squeeze(0)
    # Unfold for patches
    patches = img_tensor.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    patches = patches.permute(1, 2, 0, 3, 4).reshape(-1, 3, patch_size, patch_size)  # [49,3,32,32]
    return patches

def create_edge_index(H=7, W=7):
    src, tgt = [], []
    for i in range(H):
        for j in range(W):
            idx = i * W + j
            for di, dj in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
                ni, nj = i + di, j + dj
                if 0 <= ni < H and 0 <= nj < W:
                    src.append(idx)
                    tgt.append(ni * W + nj)
    return torch.tensor([src, tgt], dtype=torch.long)

EDGE_INDEX = create_edge_index()

In [6]:
import json
import torch.backends.cudnn as cudnn  # Add this import at top if needed
cudnn.benchmark = True  # Optimize for fixed input sizes (patches)

OUT_DIR = Path('/kaggle/working/alzheimers_dst_hgnn_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def train_baseline(model, train_loader, val_loader, epochs=EPOCHS, lr=1e-4):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    history = {'train_loss':[], 'val_loss':[], 'val_acc':[]}
    best_acc=0.0; best_state=None
    for ep in range(epochs):
        model.train(); running=0.0
        pbar = tqdm(train_loader, desc=f'Baseline Ep {ep+1} Train')
        for xb,yb,paths in pbar:
            xb,yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); out = model(xb); loss = crit(out,yb); loss.backward(); opt.step()
            running += loss.item()*xb.size(0)
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        tr_loss = running/len(train_loader.dataset)
        # val (similar, with postfix)
        model.eval(); vloss=0.0; correct=0; total=0
        pbar_val = tqdm(val_loader, desc=f'Baseline Ep {ep+1} Val', leave=False)
        with torch.no_grad():
            for xb,yb,paths in pbar_val:
                xb,yb = xb.to(DEVICE), yb.to(DEVICE)
                out = model(xb); loss = crit(out,yb)
                vloss += loss.item()*xb.size(0)
                preds = out.argmax(1)
                correct += (preds==yb).sum().item(); total += yb.size(0)
                pbar_val.set_postfix({'loss': f'{loss.item():.4f}'})
        val_loss = vloss/len(val_loader.dataset); val_acc = correct/total
        history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)
        print(f'Baseline Ep {ep+1}/{epochs} TrLoss={tr_loss:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')
        if val_acc>best_acc:
            best_acc=val_acc; best_state=model.state_dict(); torch.save(best_state, OUT_DIR/'baseline_best.pth')
    return model, history, best_state

class PatchDataset(Dataset):
    def __init__(self, subset, transform=None, patch_size=32, target_size=224):
        self.subset = subset
        self.transform = transform
        self.patch_size = patch_size
        self.target_size = target_size

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        _, label, path = self.subset[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img_tensor = self.transform(img)
        else:
            img_tensor = transforms.ToTensor()(img)
        patches = image_to_patches_torch(img_tensor, self.patch_size, self.target_size)
        return patches, label

def train_dst_hgnn(model, train_ds, val_ds, epochs=EPOCHS, lr=1e-4, patch_size=32, transform=None):
    print("Starting optimized DST-HGNN training... (Expect 1-2 mins/epoch; monitor GPU with !nvidia-smi)")
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    history = {'train_loss':[], 'val_loss':[], 'val_acc':[]}
    best_acc=0.0; best_state=None
    
    # Temp larger batch for GPU saturation (revert to 16 if OOM)
    local_batch_size = 32
    train_patch_ds = PatchDataset(train_ds, transform=transform, patch_size=patch_size)
    val_patch_ds = PatchDataset(val_ds, transform=transform, patch_size=patch_size)
    train_loader_patch = DataLoader(train_patch_ds, batch_size=local_batch_size, shuffle=True, num_workers=2, pin_memory=True)  # Parallel loading
    val_loader_patch = DataLoader(val_patch_ds, batch_size=local_batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    global EDGE_INDEX
    for ep in range(epochs):
        model.train(); running=0.0
        pbar = tqdm(train_loader_patch, desc=f'DST-HGNN Ep {ep+1} Train')
        for batch_idx, (patches, labels) in enumerate(pbar):
            patches = patches.to(DEVICE, non_blocking=True)  # Async transfer
            labels = labels.to(DEVICE, non_blocking=True)
            opt.zero_grad()
            logits, _ = model(patches, EDGE_INDEX.to(DEVICE))
            loss = crit(logits, labels)
            loss.backward(); opt.step()
            running += loss.item() * len(labels)
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'util': 'check nvidia-smi'})
        tr_loss = running / len(train_patch_ds)
        
        model.eval(); vloss=0.0; correct=0; total=0
        pbar_val = tqdm(val_loader_patch, desc=f'DST-HGNN Ep {ep+1} Val', leave=False)
        with torch.no_grad():
            for patches, labels in pbar_val:
                patches = patches.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                logits, _ = model(patches, EDGE_INDEX.to(DEVICE))
                loss = crit(logits, labels)
                vloss += loss.item() * len(labels)
                preds = logits.argmax(1)
                correct += (preds == labels).sum().item(); total += len(labels)
                pbar_val.set_postfix({'loss': f'{loss.item():.4f}'})
        val_loss = vloss / len(val_patch_ds); val_acc = correct / total
        history['train_loss'].append(tr_loss); history['val_loss'].append(val_loss); history['val_acc'].append(val_acc)
        print(f'DST-HGNN Ep {ep+1}/{epochs} TrLoss={tr_loss:.4f} ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}')
        if val_acc > best_acc:
            best_acc = val_acc; best_state = model.state_dict(); torch.save(best_state, OUT_DIR/'dst_hgnn_best.pth')
    return model, history, best_state

In [7]:
# Create baseline dataloaders that return (img,label,path)
def collate_img(batch):
    imgs = torch.stack([b[0] for b in batch])
    labels = torch.tensor([b[1] for b in batch])
    paths = [b[2] for b in batch]
    return imgs, labels, paths

train_loader_img = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_img, num_workers=2)
val_loader_img = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_img, num_workers=2)

# Baseline
baseline_model = CNNBaseline().to(DEVICE)
baseline_model, hist_base, state_base = train_baseline(baseline_model, train_loader_img, val_loader_img, epochs=EPOCHS)

# DST-HGNN (optimized)
transform_full = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225])
])

if PYG_AVAILABLE:
    dst_model = DST_HGNN_PYG(patch_dim=128, gnn_hidden=128, num_classes=NUM_CLASSES, patch_size=32).to(DEVICE)
else:
    dst_model = DST_HGNN_Sparse(patch_dim=128, gnn_hidden=128, num_classes=NUM_CLASSES, patch_size=32).to(DEVICE)

dst_model, hist_dst, state_dst = train_dst_hgnn(dst_model, train_ds, val_ds, epochs=EPOCHS, patch_size=32, transform=transform_full)

# Save histories
with open(OUT_DIR/'history_baseline.json','w') as f: json.dump(hist_base,f)
with open(OUT_DIR/'history_dst.json','w') as f: json.dump(hist_dst,f)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 176MB/s]
Baseline Ep 1 Train: 100%|██████████| 1925/1925 [02:31<00:00, 12.68it/s, loss=0.2701]


Baseline Ep 1/8 TrLoss=0.3099 ValLoss=0.1739 ValAcc=0.9314


Baseline Ep 2 Train: 100%|██████████| 1925/1925 [01:15<00:00, 25.36it/s, loss=0.3155]


Baseline Ep 2/8 TrLoss=0.0674 ValLoss=0.0430 ValAcc=0.9847


Baseline Ep 3 Train: 100%|██████████| 1925/1925 [01:11<00:00, 26.76it/s, loss=0.0336]


...

DST-HGNN Ep 6/8 TrLoss=1.2400 ValLoss=1.2139 ValAcc=0.4056


DST-HGNN Ep 7 Train: 100%|██████████| 963/963 [02:11<00:00,  7.34it/s, loss=1.1767, util=check nvidia-smi]


DST-HGNN Ep 7/8 TrLoss=1.2161 ValLoss=1.2037 ValAcc=0.4217


DST-HGNN Ep 8 Train: 100%|██████████| 963/963 [01:58<00:00,  8.15it/s, loss=1.5555, util=check nvidia-smi]
                                                                                 

DST-HGNN Ep 8/8 TrLoss=1.1996 ValLoss=1.1763 ValAcc=0.4355


In [8]:
from sklearn.preprocessing import label_binarize

# Baseline test
baseline_model.load_state_dict(state_base)
baseline_model.eval()
all_labels_b=[]; all_preds_b=[]; all_probs_b=[]; features_b=[]
with torch.no_grad():
    for img,label,path in test_ds:
        x = img.unsqueeze(0).to(DEVICE)
        out = baseline_model(x)
        prob = torch.softmax(out,dim=1).cpu().numpy()[0]
        pred = out.argmax(1).cpu().numpy()[0]
        all_labels_b.append(label); all_preds_b.append(int(pred)); all_probs_b.append(prob); features_b.append(out.cpu().numpy()[0])

rep_base = classification_report(all_labels_b, all_preds_b, target_names=CLASS_NAMES, digits=4, output_dict=True)
pd.DataFrame(rep_base).T.to_csv(OUT_DIR/'baseline_classification_report.csv')
cm_base = confusion_matrix(all_labels_b, all_preds_b)
pd.DataFrame(cm_base, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUT_DIR/'baseline_confusion.csv')

# DST-HGNN test
dst_model.load_state_dict(state_dst)
dst_model.eval()
all_labels_d=[]; all_preds_d=[]; all_probs_d=[]; features_d=[]
test_patch_ds = PatchDataset(test_ds, transform=transform_full, patch_size=32)
test_loader_patch = DataLoader(test_patch_ds, batch_size=1, shuffle=False, num_workers=2)
with torch.no_grad():
    for patches, label in test_loader_patch:
        patches = patches.to(DEVICE)
        logits, h = dst_model(patches, EDGE_INDEX.to(DEVICE))
        prob = torch.softmax(logits,dim=1).cpu().numpy()[0]
        pred = int(logits.argmax(1).cpu().numpy()[0])
        all_labels_d.append(label); all_preds_d.append(pred); all_probs_d.append(prob); features_d.append(h.cpu().numpy()[0])

rep_dst = classification_report(all_labels_d, all_preds_d, target_names=CLASS_NAMES, digits=4, output_dict=True)
pd.DataFrame(rep_dst).T.to_csv(OUT_DIR/'dst_classification_report.csv')
cm_dst = confusion_matrix(all_labels_d, all_preds_d)
pd.DataFrame(cm_dst, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(OUT_DIR/'dst_confusion.csv')

# Save confusion matrices as heatmaps
plt.figure(figsize=(6,5)); sns.heatmap(cm_dst, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('DST-HGNN Confusion Matrix'); plt.xlabel('Pred'); plt.ylabel('True'); plt.savefig(OUT_DIR/'dst_confusion.png'); plt.close()

# AUC-ROC (multiclass OVR) for DST-HGNN
labels_bin = label_binarize(all_labels_d, classes=list(range(NUM_CLASSES)))
probs_np = np.array(all_probs_d)
fpr={}; tpr={}
roc_auc={}
for i in range(NUM_CLASSES):
    fpr[i], tpr[i], _ = roc_curve(labels_bin[:,i], probs_np[:,i])
    roc_auc[i] = auc(fpr[i], tpr[i])
plt.figure()
for i in range(NUM_CLASSES):
    plt.plot(fpr[i], tpr[i], label=f'{CLASS_NAMES[i]} (AUC={roc_auc[i]:.4f})')
plt.plot([0,1],[0,1],'k--'); plt.xlabel('FPR'); plt.ylabel('TPR'); plt.legend(); plt.title('DST-HGNN ROC per class')
plt.savefig(OUT_DIR/'dst_roc.png'); plt.close()

# t-SNE on DST embeddings
feats = np.array(features_d)
if feats.shape[0] >= 5:
    ts = TSNE(n_components=2, random_state=42)
    emb2 = ts.fit_transform(feats)
    plt.figure(figsize=(6,6))
    for i,c in enumerate(CLASS_NAMES):
        idx = np.where(np.array(all_labels_d)==i)
        plt.scatter(emb2[idx,0], emb2[idx,1], label=c, alpha=0.6)
    plt.legend(); plt.title('t-SNE of DST embeddings'); plt.savefig(OUT_DIR/'dst_tsne.png'); plt.close()

In [9]:
if CAPTUM_OK:
    from captum.attr import LayerGradCam, LayerAttribution
    # pick a few test images
    sample_paths = [test_ds[i][2] for i in range(min(6, len(test_ds)))]
    target_layer = baseline_model.backbone.layer4
    gradcam = LayerGradCam(baseline_model, target_layer)
    for idx,p in enumerate(sample_paths):
        img = Image.open(p).convert('RGB')
        img_t = transform(img).unsqueeze(0).to(DEVICE)
        out = baseline_model(img_t)
        pred = int(out.argmax(1).cpu().numpy()[0])
        attributions = gradcam.attribute(img_t, target=pred)
        # convert to visualization (upsample)
        attr_upsampled = LayerAttribution.interpolate(attributions, (IMG_SIZE,IMG_SIZE))
        np_img = img_t.squeeze(0).cpu().numpy().transpose(1,2,0)
        # Normalize for display
        np_img = (np_img - np_img.min())/(np_img.max()-np_img.min())
        fig,ax = plt.subplots(1,2,figsize=(8,4))
        ax[0].imshow(np_img); ax[0].axis('off'); ax[0].set_title('Image')
        ax[1].imshow(np_img); ax[1].imshow(attr_upsampled.squeeze().cpu().numpy(), cmap='jet', alpha=0.5); ax[1].axis('off'); ax[1].set_title(f'Grad-CAM pred={CLASS_NAMES[pred]}')
        plt.savefig(OUT_DIR/f'gradcam_{idx}.png'); plt.close()
else:
    print('Captum not available; skip Grad-CAM')

Captum not available; skip Grad-CAM


In [10]:
if LIME_OK:
    explainer = lime_image.LimeImageExplainer()
    for i in range(min(4, len(test_ds))):
        path = test_ds[i][2]
        img = Image.open(path).convert('RGB')
        np_img = np.array(img)
        def predict_fn(images):
            # images: list of HxWxC arrays, need to preprocess and batch
            batch = torch.stack([transform(Image.fromarray(im)) for im in images]).to(DEVICE)
            with torch.no_grad():
                out = baseline_model(batch).cpu().numpy()
                probs = np.exp(out) / np.sum(np.exp(out), axis=1, keepdims=True)
            return probs
        explanation = explainer.explain_instance(np_img, predict_fn, top_labels=3, hide_color=0, num_samples=200)
        temp, mask = explanation.get_image_and_mask(explanation.top_labels[0], positive_only=True, num_features=5, hide_rest=False)
        plt.imsave(OUT_DIR/f'lime_{i}.png', temp)
else:
    print('LIME not available; skip LIME explanations')

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

In [11]:
# Plot training/validation loss and val acc for both
plt.figure();
plt.plot(hist_base['train_loss'], label='base_train'); plt.plot(hist_base['val_loss'], label='base_val'); plt.plot(hist_dst['train_loss'], label='dst_train'); plt.plot(hist_dst['val_loss'], label='dst_val')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('Learning Curves'); plt.savefig(OUT_DIR/'learning_curves.png'); plt.close()

plt.figure();
plt.plot(hist_base['val_acc'], label='base_val_acc'); plt.plot(hist_dst['val_acc'], label='dst_val_acc'); plt.xlabel('Epoch'); plt.ylabel('Val Acc'); plt.legend(); plt.title('Validation Accuracy'); plt.savefig(OUT_DIR/'val_acc_curve.png'); plt.close()

# Summary table (accuracy, test loss if available)
summary = {
    'model': ['baseline','dst_hgnn'],
    'val_acc': [hist_base['val_acc'][-1] if len(hist_base['val_acc'])>0 else None, hist_dst['val_acc'][-1] if len(hist_dst['val_acc'])>0 else None]
}
pd.DataFrame(summary).to_csv(OUT_DIR/'summary_table.csv', index=False)

In [12]:
print('Saved outputs to', OUT_DIR)
print('Files include: baseline_classification_report.csv, dst_classification_report.csv, dst_confusion.png, dst_roc.png, dst_tsne.png, gradcam_*.png, lime_*.png, learning_curves.png')

Saved outputs to /kaggle/working/alzheimers_dst_hgnn_outputs
Files include: baseline_classification_report.csv, dst_classification_report.csv, dst_confusion.png, dst_roc.png, dst_tsne.png, gradcam_*.png, lime_*.png, learning_curves.png
